# WhatTheFih · Stage 3 on Colab

For each of the 7,106 photos, this notebook:
1. finds the fish with GroundingDINO (a box and a score),
2. crops it (the box plus 15% on each side, made square with grey borders),
3. makes the crop's BioCLIP 2 fingerprint (768 numbers).

**Before you start**
1. In Google Drive, make a folder called **WhatTheFih** in **My Drive**. Upload **photos.zip** into it.
2. Menu: **Runtime → Change runtime type → T4 GPU → Save**.

**Run it**
1. Menu: **Runtime → Run all**.
2. When Colab asks, allow access to your Google Drive.
3. Wait about 1 to 1.5 hours. Keep this tab open.

If Colab disconnects, do **Runtime → Run all** again. The run continues where it stopped.

**At the end**, download **WhatTheFih/stage3_results.zip** (about 25 MB) from Google Drive.

### 1. Check that this session has a GPU

In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("No GPU. Menu: Runtime > Change runtime type > T4 GPU > Save. Then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

### 2. Connect Google Drive and find photos.zip

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/WhatTheFih")
PHOTOS_ZIP = DRIVE / "photos.zip"
RESULTS = DRIVE / "stage3_results"
if not PHOTOS_ZIP.exists():
    raise RuntimeError(f"Can't find {PHOTOS_ZIP}. Upload photos.zip into a folder called WhatTheFih in My Drive.")
print(f"Found photos.zip ({PHOTOS_ZIP.stat().st_size / 1e9:.2f} GB)")

### 3. Get the code from GitHub (the same code the laptop and the app use)

In [ ]:
!rm -rf /content/WhatTheFih
!git clone -q --depth 1 https://github.com/OrangeCatLoves/WhatTheFih.git /content/WhatTheFih
!git -C /content/WhatTheFih log -1 --date=short --format="Code version: %h (%cd) %s"

### 4. Install Python 3.12 and the pinned packages (a few minutes)
Same versions as the laptop, kept apart from Colab's own packages.

In [ ]:
!pip install -q uv
!rm -rf /content/venv
!uv venv -q --python 3.12 /content/venv
!uv pip install -q --python /content/venv/bin/python -r /content/WhatTheFih/requirements.txt --index-strategy unsafe-best-match
!/content/venv/bin/python -c "import torch; print('torch', torch.__version__, '| GPU works:', torch.cuda.is_available())"

### 5. Unpack the photos onto this session's own disk

In [ ]:
import shutil
import zipfile

local_zip = Path("/content/photos.zip")
shutil.copy(PHOTOS_ZIP, local_zip)  # one big copy from Drive is much faster than reading 7,000 small files
with zipfile.ZipFile(local_zip) as z:
    z.extractall("/content/WhatTheFih/data")
local_zip.unlink()
print(len(list(Path("/content/WhatTheFih/data/photos").rglob("*.*"))), "photos ready")

### 6. Find, crop and fingerprint every fish (about 1 hour)
Progress is saved to Google Drive as the run goes.

In [ ]:
!cd /content/WhatTheFih && /content/venv/bin/python scripts/detect_and_embed.py --out "/content/drive/MyDrive/WhatTheFih/stage3_results"

### 7. Pack the results for download

In [ ]:
with zipfile.ZipFile(DRIVE / "stage3_results.zip", "w") as z:
    for name in ["detections.csv", "embeddings.npz", "run_info.json"]:
        z.write(RESULTS / name, name)
size_mb = (DRIVE / "stage3_results.zip").stat().st_size / 1e6
drive.flush_and_unmount()  # make sure everything has reached Google Drive
print(f"Done. In Google Drive, download WhatTheFih/stage3_results.zip ({size_mb:.0f} MB).")